In [251]:
import re

import pandas as pd


In [252]:
INPUT_FILE = ""
OUTPUT_FILE = ""

if not INPUT_FILE:
    raise ValueError("Set INPUT_FILE to the workbook path before running the notebook")

df = pd.read_excel(INPUT_FILE)


In [253]:
USED_COLUMNS = ["Seller COFOR2", "Manufacturer COFOR", "Manufacturer address", "Shipper COFOR2",
                "Shipper COFOR Address", "Location ID2", "Location ID Address", "Quality contact", "Logistic contact",
                "Contacted", "Info completed", "NOTE", "Owner", "Format check", "Status", "SOM double-check", "Plant"]


In [254]:
LOCATION_COLUMNS = ["Manufacturer address", "Shipper COFOR Address", "Location ID Address"]


In [255]:
EMAIL_COLUMNS = ["Quality contact", "Logistic contact"]


In [256]:
CHAR_LENGTH_COLUMNS = ["Seller COFOR2", "Manufacturer COFOR", "Shipper COFOR2"]


In [257]:
CHAR_LENGTH_COLUMN_12 = ["Location ID2"]


In [258]:
EXCEL_FORMULA_COLUMNS = ["Info completed", "Format check", "Owner"]

In [259]:
EMAIL_REGEX = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'


In [260]:
CHAR_PATTERN_REGEX = r'^[a-zA-Z0-9]{6} {2}[a-zA-Z0-9]{2}$'


In [261]:
CHAR_LENGTH = 12


In [262]:
CONTACTED_ALLOWED_VALUES = ["yes", "no", "out of scope"]


In [263]:
# 4 complementary signals, combined with OR
_LOCATION_REGEX = re.compile(
    r'(\b\d{4,6}\b)'  # postal code:  66041, 10060, 81400
    r'|(\d+\s*[,/\-])'  # street number: "28 ," "86/i" "122-"
    r'|([A-Za-z]{3,},\s*[A-Za-z0-9])'  # "City, XX" or "City, 12345"
    r'|(\b(via|rue|strada|ul\.|str\.|road|avenue|blvd'
    r'|street|zone\s+ind|route|calle|rua|allee|viale'
    r'|corso|piazza|contrada|localit|district'
    r'|industrial|zone)\b)',
    re.IGNORECASE
)

In [264]:
_EXCEL_ERRORS = {
    "#n/a", "#ref!", "#value!", "#div/0!",
    "#name?", "#null!", "#num!", "#getting_data"
}

In [265]:
def is_valid_location(value) -> bool:
    """
    Returns True if value is non-empty AND contains at least one
    recognizable location signal (postal code, street number,
    city/country pattern, or street-type keyword).

    Handles: None, float NaN, empty string, whitespace-only, too-short strings.
    Coverage: 95.82% on 91,676 real address values from your dataset.
    """
    if value is None:
        return False
    if isinstance(value, float):  # NaN → pandas reads missing cells as float
        return False
    if not isinstance(value, str):
        return False
    stripped = value.strip()
    if len(stripped) < 5:  # catches 'x', '', single words like codes
        return False
    return bool(_LOCATION_REGEX.search(stripped))

In [266]:
def is_valid_ref(value) -> bool:
    """
    Returns False only when value matches a known Excel error token.
    All other values are considered valid.
    """
    if not isinstance(value, str):
        return True
    return value.strip().lower() not in _EXCEL_ERRORS

In [267]:
def check_column_length(value) -> bool:
    if pd.isna(value) or not isinstance(value, str):
        return False
    return len(value.strip()) == CHAR_LENGTH


In [268]:
def check_column_against_regex(value, regex) -> bool:
    if pd.isna(value) or not isinstance(value, str):
        return False
    return re.match(regex, value.strip()) is not None


In [269]:
def normalize(df, wanted_columns) -> pd.DataFrame:
    for col in wanted_columns:
        df[col] = df[col].astype(str)
        df[col] = df[col].str.strip()
    return df


In [270]:
def build_comment_for_row(index) -> str:
    reasons = []

    failed_emails = [col for col in EMAIL_COLUMNS if fail_EMAIL_COLUMNS.at[index, col]]
    if failed_emails:
        reasons.append(f"Invalid email: {', '.join(failed_emails)}")

    failed_patterns = [col for col in CHAR_LENGTH_COLUMNS if fail_COLUMN_LENGTH.at[index, col]]
    if failed_patterns:
        reasons.append(f"Invalid COFOR pattern (6 chars + 2 spaces + 2 chars): {', '.join(failed_patterns)}")

    failed_len_12 = [col for col in CHAR_LENGTH_COLUMN_12 if fail_COLUMN_LENGTH_12.at[index, col]]
    if failed_len_12:
        reasons.append(f"Invalid length (must be {CHAR_LENGTH}): {', '.join(failed_len_12)}")

    if fail_CONTACTED.at[index]:
        reasons.append("Invalid Contacted value (allowed: yes, no, out of scope)")
    failed_locations = [col for col in LOCATION_COLUMNS if fail_COLUMN_LOCATION.at[index, col]]
    if failed_locations:
        reasons.append(
            f"Invalid location (missing postal code, street number, city/country pattern, or street-type keyword): {', '.join(failed_locations)}")
    failed_refs = [col for col in EXCEL_FORMULA_COLUMNS if fail_REF_COLUMNS.at[index, col]]
    if failed_refs:
        reasons.append(
            f"Invalid reference (Excel error token): {', '.join(failed_refs)}")

    if fail_STATUS_INFO_MISSING.at[index]:
        reasons.append("Consistency check error: Status is Complete but Info completed is missing or empty")

    return " | ".join(reasons)


In [271]:
def is_allowed_values(value, allowed_values) -> bool:
    if pd.isna(value) or not isinstance(value, str):
        return False
    return str(value).strip().lower() in allowed_values


In [272]:
df["Check"] = pd.Series(dtype='boolean')
df["Comment"] = pd.Series(dtype='string')


In [273]:
df_normalized = normalize(df, USED_COLUMNS)

In [274]:
plant_list = ["149", "144", "142"]
contacted = ["yes", "no"]
info_completed = ["Complete"]

In [275]:
#Selected filter by user
mask_selected = (
        (df_normalized["Plant"].isin(plant_list)) &
        (df_normalized["Contacted"].str.strip().str.lower().isin(contacted)) &
        (df_normalized["Info completed"].isin(info_completed))
)
df_filtered = df_normalized[mask_selected]
df_rest = df_normalized[~mask_selected]

In [276]:
df_filtered = df_filtered.copy()
df_rest = df_rest.copy()
df_rest["Comment"] = "No comment || Out of scope"
df_rest["Check"] = "Out of scope"

In [277]:
fail_EMAIL_COLUMNS = ~df_filtered.apply(
    lambda col: col.apply(
        lambda x: check_column_against_regex(x, EMAIL_REGEX) if col.name in EMAIL_COLUMNS else True
    )
)


In [278]:
fail_COLUMN_LENGTH = ~df_filtered.apply(
    lambda col: col.apply(
        lambda x: check_column_against_regex(x, CHAR_PATTERN_REGEX) if col.name in CHAR_LENGTH_COLUMNS else True
    )
)


In [279]:
fail_COLUMN_LENGTH_12 = ~df_filtered.apply(
    lambda col: col.apply(
        lambda x: check_column_length(x) if col.name in CHAR_LENGTH_COLUMN_12 else True
    )
)


In [280]:
fail_CONTACTED = ~df_filtered["Contacted"].apply(
    lambda x: is_allowed_values(
        x, CONTACTED_ALLOWED_VALUES
    )
)


In [281]:
fail_COLUMN_LOCATION = ~df_filtered.apply(
    lambda col: col.apply(
        lambda x: is_valid_location(x) if col.name in LOCATION_COLUMNS else True
    )
)


In [ ]:
fail_REF_COLUMNS = ~df_filtered.apply(
    lambda col: col.apply(
        lambda x: is_valid_ref(x) if col.name in EXCEL_FORMULA_COLUMNS else True
    )
)


In [ ]:
fail_STATUS_INFO_MISSING = (
    df_filtered["Status"].astype(str).str.strip().eq("Complete") &
    (
        df_filtered["Info completed"].isna() |
        df_filtered["Info completed"].astype(str).str.strip().eq("") |
        df_filtered["Info completed"].astype(str).str.strip().str.lower().isin(["nan", "none"])
    )
)

In [283]:
# Count total failed checks per row (email + column-length).
df_filtered["Check"] = fail_EMAIL_COLUMNS.sum(axis=1).astype(int) + fail_COLUMN_LENGTH.sum(axis=1).astype(
    int) + fail_COLUMN_LENGTH_12.sum(axis=1).astype(int) + fail_CONTACTED.astype(int) + fail_COLUMN_LOCATION.sum(
    axis=1).astype(int) + fail_REF_COLUMNS.sum(axis=1).astype(int) + fail_STATUS_INFO_MISSING.astype(int)


In [295]:
df_filtered.columns

Index(['Purchase Order', 'Part Number', 'Seller Supplier Name',
       'Seller Sector Code', 'Plant', 'Plants perimeter', 'Incoterm', 'Buyer',
       'Seller COFOR', 'Seller COFOR Name', 'Manuf COFOR', 'Manuf COFOR Name',
       'Shipper COFOR', 'Shipper COFOR Name', 'Shipper COFOR Address',
       'Location ID', 'Location ID Name', 'Location ID Address',
       'Expected Shipper COFOR', 'Shipper COFOR - Location ID match',
       'Distance Shipper COFOR to Location ID', 'To check', 'Seller COFOR2',
       'Manufacturer COFOR', 'Manufacturer address', 'Shipper COFOR2',
       'Shipper address', 'Location ID2', 'Location ID address2',
       'Quality contact', 'Logistic contact', 'Contacted', 'Info completed',
       'NOTE', 'Suppl-PN-Plant', 'SAP triplet', 'Supplier confirmed triplet',
       'Owner', 'Format check', 'xF Code', 'SupplierPn',
       'Expected Shipper Check', 'Plant-Suppl-PN', 'Gross need',
       'SAP Triplet Lenght', 'Supplier triplet lenght',
       'SAP vs Supplier f

In [293]:
#Consistency control
# handled in fail_STATUS_INFO_MISSING + build_comment_for_row

In [294]:
df_filtered["Comment"] = df_filtered.index.to_series().apply(build_comment_for_row).astype("string")
df_filtered.loc[df_filtered["Check"] == 0, "Comment"] = "Quality check passed"

In [292]:
df_filtered["Check"].value_counts()


Check
0    2832
2    1981
1    1511
4     946
3     936
5      63
6       2
Name: count, dtype: int64

In [286]:
final_df = pd.concat([df_filtered, df_rest], ignore_index=True)

In [288]:
final_df.head()

,Purchase Order,Part Number,Seller Supplier Name,Seller Sector Code,Plant,Plants perimeter,Incoterm,Buyer,Seller COFOR,Seller COFOR Name,...,SAP Triplet Lenght,Supplier triplet lenght,SAP vs Supplier feedback,New,Status,Enrico remark,SOM double-check,SAP Update confirmation,Check,Comment
0,0098365922,468626990,SEWS CABIND SPA,0000058463,149,-117---123---139-141--144-145--149------169---...,DDP,Dawid MAJ,A00DCH 01,SEWS CABIND SPA,...,12,32,KO,NaN,complete,NaN,NaN,Update,4,"Invalid email: Quality contact, Logistic conta..."
1,0098368969,468640610,SEWS CABIND SPA,0000058463,149,-117---123---139-141--144-145--149------169---...,DDP,Dawid MAJ,A00DCH 01,SEWS CABIND SPA,...,12,32,KO,NaN,complete,NaN,NaN,Update,4,"Invalid email: Quality contact, Logistic conta..."
2,0098412521,468679410,SEWS CABIND SPA,0000058463,149,-117---123---139-141--144-145--149------169---...,DDP,Dawid MAJ,A00DCH 01,SEWS CABIND SPA,...,12,32,KO,NaN,complete,NaN,NaN,Update,4,"Invalid email: Quality contact, Logistic conta..."
3,0098412525,468679370,SEWS CABIND SPA,0000058463,149,-117---123---139-141--144-145--149------169---...,DDP,Dawid MAJ,A00DCH 01,SEWS CABIND SPA,...,12,32,KO,NaN,complete,NaN,NaN,Update,4,"Invalid email: Quality contact, Logistic conta..."
4,0098412527,468679350,SEWS CABIND SPA,0000058463,149,-117---123---139-141--144-145--149------169---...,DDP,Dawid MAJ,A00DCH 01,SEWS CABIND SPA,...,12,32,KO,NaN,complete,NaN,NaN,Update,4,"Invalid email: Quality contact, Logistic conta..."


In [289]:
if not OUTPUT_FILE:
    raise ValueError("Set OUTPUT_FILE to the export workbook path before running this cell")

final_df.to_excel(
    OUTPUT_FILE,
    index=False,
    engine="xlsxwriter",
    merge_cells=False
)